In [ ]:
#| default_exp llm_core.call_llm

In [ ]:
#| export
import re
from typing import Any, List, Literal, Optional, overload, TypedDict, Union, Any, Protocol, runtime_checkable

from openai import OpenAI
import lmstudio


import time
from typing import List, Optional, Any, Tuple

In [ ]:
from fastcore.test import *
import time

from unittest.mock import patch

This module handles the general logic for calling llm's and processing their outputs.

## Separating thoughts from actual response for reasoning models

In [ ]:
#| export
def separate_thoughts(
        raw_content: str
        ) -> tuple[Optional[str], str]:
    """
    Separates model 'reasoning' from the actual answer.
    Handles <think>, <thought>, and [THOUGHT] tags.
    """
    # 1. Define common patterns for thinking blocks
    # This regex looks for <think>...</think> or  (case insensitive)
    tag_pattern = r"<(think|thought)>([\s\S]*?)<\/\1>"
    
    match = re.search(tag_pattern, raw_content, re.IGNORECASE)
    
    if match:
        thoughts = match.group(2).strip()
        # Remove the thinking block from the main content
        answer = re.sub(tag_pattern, "", raw_content, flags=re.IGNORECASE).strip()
        return thoughts, answer
    
    # 2. Fallback for models that don't use tags but use a header
    if "THOUGHTS:" in raw_content.upper():
        parts = re.split(r"THOUGHTS:", raw_content, flags=re.IGNORECASE)
        # Assuming format: THOUGHTS: [logic] ANSWER: [result]
        if "ANSWER:" in parts[1].upper():
            sub_parts = re.split(r"ANSWER:", parts[1], flags=re.IGNORECASE)
            return sub_parts[0].strip(), sub_parts[1].strip()
            
    # 3. If no markers found, return everything as the answer
    return None, raw_content.strip()

The ```separate_thoughts``` function is used to extract the thought process and the final output of reasoning models (e.g. `DeepSeek-R1`, `Qwen3`, `GPT-5.2 Thinking`, etc.)

In [ ]:
ex1 = """First, analyze the problem.<think>Step 1: Identify key components. Step 2: Validate inputs.</think> The final answer is 42."""

thoughts1, answer1 = separate_thoughts(ex1)
test_eq(thoughts1, "Step 1: Identify key components. Step 2: Validate inputs.")
test_eq(answer1, "First, analyze the problem. The final answer is 42.")

In [ ]:
ex2 = """<THOUGHT>
Mathematical reasoning: check base cases first, then induction step.
Edge case x=0 gives y=1.
</THOUGHT>
Final computation: ∫[0,1] x² dx = 1/3"""

thoughts2, answer2 = separate_thoughts(ex2)
test_eq(thoughts2, "Mathematical reasoning: check base cases first, then induction step.\nEdge case x=0 gives y=1.")
test_eq(answer2, "Final computation: ∫[0,1] x² dx = 1/3")

In [ ]:
ex3 = """THOUGHTS: Gal(L/K) is defined via étale cohomology. Verify separability first.
ANSWER: The Galois group Gal(L/K) is finite of order [L:K]."""

thoughts3, answer3 = separate_thoughts(ex3)
test_eq(thoughts3, "Gal(L/K) is defined via étale cohomology. Verify separability first.")
test_eq(answer3, "The Galois group Gal(L/K) is finite of order [L:K].")

In [ ]:
#| hide
from fastcore.test import *

# Tag extraction works
test_eq(separate_thoughts("<think>reasoning</think>")[0], "reasoning")
test_eq(separate_thoughts("<THOUGHT>test</thought>")[0], "test")

# Answer extraction - EXACT original behavior
test_eq(separate_thoughts("A<think>B</think>C")[1], "AC")
test_eq(separate_thoughts("A <think>B</think> C")[1], "A  C")
test_eq(separate_thoughts("A\n<think>B</think>\nC")[1], "A\n\nC")

# Multiple tags - re.sub removes FIRST match + .strip()
multi = "<think>first</think>text<think>second</think>"
test_eq(separate_thoughts(multi)[0], "first")
test_eq(separate_thoughts(multi)[1], "text")  # ← Fixed: .strip() removes trailing tag

# No tags
test_eq(separate_thoughts("plain text")[0], None)
test_eq(separate_thoughts("plain text")[1], "plain text")

# Malformed tags
test_eq(separate_thoughts("<think>no close")[0], None)
test_eq(separate_thoughts("<think>no close")[1], "<think>no close")

# THOUGHTS:ANSWER: fallback
test_eq(separate_thoughts("THOUGHTS: abc ANSWER: def"), ("abc", "def"))

# Empty cases
test_eq(separate_thoughts("<think></think>")[0], "")
test_eq(separate_thoughts("")[1], "")
test_eq(separate_thoughts("   ")[1], "")

## Handling different model API/libraries

In [ ]:
#| export

@runtime_checkable
class LLMProvider(Protocol):
    """Structural requirement for a model to be used in this script."""
    def respond(self, payload: dict, config: Optional[dict] = None) -> Any: ...
    @property
    def chat(self) -> Any: ...

# The Type Alias: This is what the user sees in their IDE hover-text.
# It explicitly lists the intended classes + our generic Protocol.
SupportedLLM = Union[lmstudio.LLM, OpenAI, LLMProvider]

In [ ]:
#| export
def smart_truncate(
    text: str, 
    model: SupportedLLM,  # Using the alias here
    max_context: int, 
    reserved_tokens: int, 
    verbose: bool = False
) -> str:
    r"""Truncates `text` based on a herustic (3 chars per token) to prevent
    context window overflow."""
    # Note: 'hasattr' still works great here for logic branching
    if hasattr(model, 'tokenize'):
        available_tokens = max_context - reserved_tokens
        char_limit = available_tokens * 3 
    else:
        available_tokens = max_context - reserved_tokens
        char_limit = available_tokens * 3
        
    if len(text) > char_limit:
        if verbose: print(f"Warning: Text truncated to ~{char_limit} chars.")
        return text[:char_limit] + "..."
    return text


In [ ]:
# Example
class MockModel: tokenize = True
text = "This is a very long string of text."
truncated = smart_truncate(text, MockModel(), max_context=10, reserved_tokens=5)
print(truncated)

This is a very ...


In [ ]:
#| hide
from fastcore.test import *

# Test basic truncation
test_eq(smart_truncate("Hello World", None, 10, 8), "Hello ...") # 2 tokens * 3 = 6 chars
# Test no truncation needed
test_eq(smart_truncate("Short", None, 100, 10), "Short")

## Calling the LLM and handling its response

In [ ]:
#| export
def _parse_groq_limits(headers: Any) -> dict:
    """Extracts Groq-specific rate limit info from response headers."""
    return {
        "remaining_requests": headers.get("x-ratelimit-remaining-requests"),
        "remaining_tokens": headers.get("x-ratelimit-remaining-tokens"),
        "reset_requests": headers.get("x-ratelimit-reset-requests"),
        "reset_tokens": headers.get("x-ratelimit-reset-tokens"),
        "retry_after": headers.get("retry-after")
    }

In [ ]:
#| export
def _extract_headers(headers: Any) -> dict:
    """Safely extracts rate limits; returns empty dict if not present."""
    if not headers: return {}
    # Use .get() to avoid KeyErrors if these aren't Groq headers
    return {
        "rem_req": headers.get("x-ratelimit-remaining-requests"),
        "rem_tok": headers.get("x-ratelimit-remaining-tokens"),
        "reset": headers.get("x-ratelimit-reset-requests")
    }

In [ ]:
#| hide
# Test Header Extraction
standard_headers = {"Content-Type": "application/json"}
groq_headers = {
    "x-ratelimit-remaining-tokens": "5000",
    "x-ratelimit-reset-requests": "10s"
}

# Standard headers should return empty values for limits, not crash
test_eq(_extract_headers(standard_headers).get('rem_tok'), None)

# Groq headers should map correctly
extracted = _extract_headers(groq_headers)
test_eq(extracted['rem_tok'], "5000")
test_eq(extracted['reset'], "10s")

# Null case
test_eq(_extract_headers(None), {})

In [ ]:
#| export
def _get_input_metrics(messages: List[dict]) -> Tuple[float, str, int]:
    """Returns (perf_counter, timestamp_string, character_count)."""
    return time.perf_counter(), time.strftime("%H:%M:%S"), sum(len(m['content']) for m in messages)

In [ ]:
#| export

def _handle_openai_call(model: Any, messages: List[dict], config: dict) -> Tuple[str, Optional[float], Any]:
    """Handles OpenAI streaming with fallback and safe limit extraction."""
    full_content, ttft, start_perf, usage = "", None, time.perf_counter(), None
    try:
        # We use with_raw_response to wrap the stream so we can peek at headers
        raw = model.chat.completions.with_raw_response.create(
            messages=messages, stream=True, stream_options={"include_usage": True}, **config
        )
        limits = _extract_headers(raw.headers) # Capture headers immediately
        for chunk in raw.parse():
            if not ttft and chunk.choices and chunk.choices[0].delta.content:
                ttft = time.perf_counter() - start_perf
            if chunk.choices and chunk.choices[0].delta.content:
                full_content += chunk.choices[0].delta.content
            if chunk.usage: usage = chunk.usage
        if usage: usage.limits = limits # Attach limits to usage
        return full_content, ttft, usage
    except Exception:
        raw_fallback = model.chat.completions.with_raw_response.create(
            messages=messages, stream=False, **config
        )
        res = raw_fallback.parse()
        res.usage.limits = _extract_headers(raw_fallback.headers)
        return res.choices[0].message.content, None, res.usage

# def _handle_openai_call(model: Any, messages: List[dict], config: dict) -> Tuple[str, Optional[float], Any]:
#     """Handles OpenAI streaming with a safe fallback to standard calls."""
#     full_content, ttft, start_perf = "", None, time.perf_counter()
#     try:
#         response = model.chat.completions.create(messages=messages, stream=True, 
#                                                  stream_options={"include_usage": True}, **config)
#         for chunk in response:
#             if not ttft and chunk.choices and chunk.choices[0].delta.content:
#                 ttft = time.perf_counter() - start_perf
#             if chunk.choices and chunk.choices[0].delta.content:
#                 full_content += chunk.choices[0].delta.content
#             if chunk.usage: return full_content, ttft, chunk.usage
#     except Exception:
#         res = model.chat.completions.create(messages=messages, stream=False, **config)
#         return res.choices[0].message.content, None, res.usage
#     return full_content, ttft, None

In [ ]:
#| export



def _handle_lms_call(model: Any, messages: List[dict], config: dict) -> Tuple[str, Optional[float], Any]:
    """Handles LM Studio streaming with a safe fallback."""
    full_content, ttft, start_perf = "", None, time.perf_counter()
    lms_config = {**config, "maxTokens": config.get("max_tokens", 1024)}
    try:
        stream = model.respond({"messages": messages}, config=lms_config, stream=True)
        for chunk in stream:
            if ttft is None: ttft = time.perf_counter() - start_perf
            full_content += getattr(chunk, 'content', "")
        return full_content, ttft, getattr(stream, 'usage', {})
    # except Exception:
    except Exception as e:
        # if verbose:
        #     print(f"[Debug] Streaming failed because: {e}")
        # ... rest of fallback code ...
        res = model.respond({"messages": messages}, config=lms_config)
        return getattr(res, 'content', str(res)), None, getattr(res, 'usage', {})

# def call_llm(
#         model: SupportedLLM,
#         messages: List[dict],
#         config: Optional[dict] = None,
#         verbose: bool = False) -> str:
#     conf = {"temperature": 0.1, "max_tokens": 1024, **(config or {})}
#     start_p, start_t, in_chars = _get_input_metrics(messages)
    
#     if hasattr(model, 'respond'):
#         out, ttft, usage = _handle_lms_call(model, messages, conf)
#     elif hasattr(model, 'chat'):
#         m_name = conf.pop("model_name", "gpt-4o")
#         out, ttft, usage = _handle_openai_call(model, messages, {"model": m_name, **conf})
#     else:
#         raise ValueError("Unsupported model interface.")

#     dur = time.perf_counter() - start_p
#     if verbose:
#         u = usage if isinstance(usage, dict) else getattr(usage, '__dict__', {})
#         tps = (u.get('completion_tokens', 0) or getattr(usage, 'completion_tokens', 0)) / dur if dur > 0 else 0
#         print(f"\n[Verbose] In: {in_chars}c | Out: {len(out)}c | Start: {start_t} | End: {time.strftime('%H:%M:%S')}")
#         print(f"[Verbose] Total: {dur:.2f}s | TTFT: {f'{ttft:.2f}s' if ttft else 'N/A'} | TPS: {tps:.2f}\n")
    
#     return out.strip()

In [ ]:
#| export
def _log_llm_stats(
        out: str,
        in_c: int,
        dur: float,
        ttft: float,
        usage: Any,
        start_t: str):
    """Prints performance and usage metrics."""
    u = usage if isinstance(usage, dict) else getattr(usage, '__dict__', {})
    comp_tokens = u.get('completion_tokens', 0) or getattr(usage, 'completion_tokens', 0)
    tps = comp_tokens / dur if dur > 0 else 0
    
    print(f"\n[LLM] In: {in_c}c | Out: {len(out)}c | Time: {start_t} -> {time.strftime('%H:%M:%S')}")
    print(f"[LLM] Dur: {dur:.2f}s | TTFT: {f'{ttft:.2f}s' if ttft else 'N/A'} | TPS: {tps:.2f}")
    
    if hasattr(usage, 'limits') and usage.limits.get('rem_tok'):
        print(f"[Limits] Remaining Tokens: {usage.limits['rem_tok']} | Reset: {usage.limits['reset']}")

In [ ]:
#| export
def call_llm(
    model: 'SupportedLLM',
    messages: List[dict],
    config: Optional[dict] = None,
    verbose: bool = False,
    return_usage: bool = False
) -> str | Tuple[str, Any]:
    """Calls the LLM and optionally returns usage metadata."""
    conf = {"temperature": 0.1, "max_tokens": 1024, **(config or {})}
    start_p, start_t, in_chars = _get_input_metrics(messages)
    
    if hasattr(model, 'respond'):
        out, ttft, usage = _handle_lms_call(model, messages, conf)
    elif hasattr(model, 'chat'):
        m_name = conf.pop("model_name", "gpt-4o")
        out, ttft, usage = _handle_openai_call(model, messages, {"model": m_name, **conf})
    else:
        raise ValueError("Unsupported model interface.")

    dur = time.perf_counter() - start_p
    if verbose:
        _log_llm_stats(out, in_chars, dur, ttft, usage, start_t)
    
    return (out.strip(), usage) if return_usage else out.strip()

In [ ]:
# Example Mocking LM Studio
class MockLMS: 
    def respond(self, p, config=None): return "Direct Response String"

messages = [{"role": "user", "content": "Hi"}]
print(call_llm(MockLMS(), messages))

Direct Response String


In [ ]:
#| hide
# 1. Fix the Mock names and make them robust
class MockLMS:
    """Simulates a model that returns a simple string or object."""
    def respond(self, payload, config=None, stream=False):
        # Your helper expects an iterable if stream=True
        if stream:
            class Chunk: content = "Object Response"
            return [Chunk()] 
        
        class Res: 
            content = "Object Response"
            usage = {"completion_tokens": 10}
        return Res()

# 2. Update the tests
# Ensure the mock name matches: MockLMS()
test_eq(call_llm(MockLMS(), [{"role":"user", "content":"hi"}]), "Object Response")

# 3. Ensure the failure test provides the expected input structure
test_fail(lambda: call_llm("NotAModel", [{"role":"user", "content":"hi"}]), contains="Unsupported model interface")

In [ ]:
#| hide
class MockStreamingModel:
    def respond(self, payload, config=None, stream=False):
        class Chunk:
            def __init__(self, content): self.content = content
        
        # This class acts as the 'Prediction' object
        class Prediction:
            def __init__(self):
                self.usage = {"completion_tokens": 2}
            def __iter__(self):
                # This makes the object 'Loop-able'
                time.sleep(0.1) # TTFT delay
                yield Chunk("Streaming ")
                yield Chunk("Response")

        if stream:
            return Prediction()
        
        # Non-streaming fallback
        class Res:
            content = "Static Response"
            usage = {"completion_tokens": 2}
        return Res()

In [ ]:
#| hide
class MockUsage:
    def __init__(self):
        self.completion_tokens = 5
        self.limits = {"rem_tok": "100"}

class MockModelWithUsage:
    """Mocks an OpenAI-like model that provides usage data."""
    def respond(self, payload, config=None, stream=False):
        class Res:
            content = "Usage Test"
            usage = MockUsage()
        return Res()

messages = [{"role": "user", "content": "test"}]
model = MockModelWithUsage()

# Test 1: Default behavior (returns str)
res_str = call_llm(model, messages, return_usage=False)
test_is(type(res_str), str)
test_eq(res_str, "Usage Test")

# Test 2: Return Usage behavior (returns Tuple)
res_tuple = call_llm(model, messages, return_usage=True)
test_is(type(res_tuple), tuple)
test_eq(len(res_tuple), 2)
test_eq(res_tuple[0], "Usage Test")
test_eq(res_tuple[1].completion_tokens, 5)

In [ ]:
#| hide
class MockRawResponse:
    """Simulates the object returned by .with_raw_response.create()"""
    def __init__(self):
        self.headers = {"x-ratelimit-remaining-tokens": "999"}
    def parse(self):
        class Choice:
            message = type('obj', (object,), {'content': "Fallback result"})
        class Usage:
            completion_tokens = 10
        return type('obj', (object,), {'choices': [Choice()], 'usage': Usage()})

# Test that limits are attached in the fallback/non-streaming path
@patch('__main__._extract_headers')
def test_metadata_attachment(mock_ext):
    mock_ext.return_value = {"rem_tok": "999"}
    # This simulates the internal behavior of _handle_openai_call's except block
    raw_fallback = MockRawResponse()
    res = raw_fallback.parse()
    res.usage.limits = _extract_headers(raw_fallback.headers)
    
    test_eq(res.usage.limits['rem_tok'], "999")
    test_eq(res.usage.completion_tokens, 10)

test_metadata_attachment()

In [ ]:
#| hide
# Test logger with dictionary (LM Studio style)
dict_usage = {"completion_tokens": 10, "limits": {"rem_tok": "50", "reset": "1s"}}
# Should not raise error
_log_llm_stats("out", 10, 1.0, 0.1, dict_usage, "12:00:00")

# Test logger with object (OpenAI style)
class ObjUsage:
    completion_tokens = 20
    limits = {"rem_tok": "100", "reset": "2s"}
# Should not raise error
_log_llm_stats("out", 10, 1.0, 0.1, ObjUsage(), "12:00:00")


[LLM] In: 10c | Out: 3c | Time: 12:00:00 -> 22:51:08
[LLM] Dur: 1.00s | TTFT: 0.10s | TPS: 10.00

[LLM] In: 10c | Out: 3c | Time: 12:00:00 -> 22:51:08
[LLM] Dur: 1.00s | TTFT: 0.10s | TPS: 20.00
[Limits] Remaining Tokens: 100 | Reset: 2s


In [ ]:
#| export
class LLMResponse(TypedDict):
    r"""
    A `TypedDict` representing a processed response with separate logic and final output.

    Output of `process_llm_response` when `return_thoughts=True`
    """
    thoughts: str
    output: str

In [ ]:
#| export
# Overload 1: If return_thoughts is True, return a dict
@overload
def process_llm_response(raw_text: str, return_thoughts: Literal[True]) -> LLMResponse: ...

# Overload 2: If return_thoughts is False (default), return a str
@overload
def process_llm_response(raw_text: str, return_thoughts: Literal[False] = False) -> str: ...

# The actual implementation (type hints here are usually more generic)
def process_llm_response(
    raw_text: str,
    return_thoughts: bool = False
) -> str | LLMResponse:
    """Separates thoughts and returns either a string or an `LLMResponse` dict."""
    if not raw_text: 
        return {"thoughts": "", "output": ""} if return_thoughts else ""
        
    thoughts, clean_answer = separate_thoughts(raw_text)
    
    if return_thoughts:
        # Cast to LLMResponse or just return; TypedDict validates the keys
        return {"thoughts": thoughts or "", "output": clean_answer}
    
    return clean_answer

In [ ]:
# Example
raw = "<think>Calculating 2+2</think>The answer is 4."
print(f"String only: {process_llm_response(raw)}")
print(f"With thoughts: {process_llm_response(raw, return_thoughts=True)}")

String only: The answer is 4.
With thoughts: {'thoughts': 'Calculating 2+2', 'output': 'The answer is 4.'}


In [ ]:
#| hide
# Note: separate_thoughts must be defined in the namespace for these to pass
raw_input = "<think>logic</think>answer"

# Test string return (default)
test_eq(process_llm_response(raw_input), "answer")

# Test Dict return
processed = process_llm_response(raw_input, return_thoughts=True)
test_is(type(processed), dict)
test_eq(processed['thoughts'], "logic")
test_eq(processed['output'], "answer")

# Test empty thoughts fallback
test_eq(process_llm_response("just answer", True)['thoughts'], "")